**CI twin of `ch11-random-forests.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from collections import Counter
import numpy as np

df = load_csv("penguins").dropna(subset=["bill_length_mm",
                                         "flipper_length_mm"])
n = len(df)

rng = np.random.default_rng(0)
sample_idx = rng.integers(0, n, n)          # n draws, with replacement
counts = Counter(sample_idx.tolist())

unique = len(counts)
repeats = sum(1 for c in counts.values() if c >= 2)
print(f"colony size: {n}")
print(f"unique birds drawn:   {unique}  ({unique / n:.0%})")
print(f"birds drawn 2+ times: {repeats}")
print(f"birds left out:       {n - unique}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

feats = ["flipper_length_mm", "bill_length_mm"]
Xtr, Xte, ytr, yte = train_test_split(
    df[feats], df["species"], test_size=0.25, random_state=42,
    stratify=df["species"])

probe = Xte.iloc[[0]]   # one held-out bird: flipper 200 mm, bill 42 mm
votes = []
for seed in range(5):
    rng = np.random.default_rng(seed)
    bidx = rng.integers(0, len(Xtr), len(Xtr))
    t = DecisionTreeClassifier(random_state=0).fit(
        Xtr.iloc[bidx], ytr.iloc[bidx])
    root = feats[t.tree_.feature[0]]
    vote = t.predict(probe)[0]
    votes.append(vote)
    print(f"tree {seed}: asks first about {root:<18} -> votes {vote}")
print(f"\ncrowd verdict: {Counter(votes).most_common(1)[0][0]}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

lone = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
forest = RandomForestClassifier(n_estimators=100, oob_score=True,
                                random_state=0).fit(Xtr, ytr)

print(f"lone full-depth tree: {accuracy_score(yte, lone.predict(Xte)):.3f}")
print(f"forest of 100:        {accuracy_score(yte, forest.predict(Xte)):.3f}")

In [ ]:
tree_scores, forest_scores = [], []
for seed in range(5):
    sub = Xtr.sample(n=150, random_state=seed)
    sub_y = ytr.loc[sub.index]
    t = DecisionTreeClassifier(random_state=0).fit(sub, sub_y)
    f = RandomForestClassifier(n_estimators=100,
                               random_state=0).fit(sub, sub_y)
    tree_scores.append(round(accuracy_score(yte, t.predict(Xte)), 3))
    forest_scores.append(round(accuracy_score(yte, f.predict(Xte)), 3))

print(f"lone tree: {tree_scores}  spread {max(tree_scores) - min(tree_scores):.3f}")
print(f"forest:    {forest_scores}  spread {max(forest_scores) - min(forest_scores):.3f}")

In [ ]:
print(f"OOB estimate: {forest.oob_score_:.3f}")
print(f"actual held-out score: 1.000")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

homes = load_csv("california-housing-sample")
hfeats = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
          "Population", "AveOccup", "Latitude", "Longitude"]

scores = -cross_val_score(
    RandomForestRegressor(n_estimators=100, random_state=0),
    homes[hfeats], homes["MedHouseVal"],
    cv=5, scoring="neg_mean_absolute_error")
print(f"forest CV MAE: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"linear (Ch4):  0.558 ± 0.031")

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=0)
model.fit(Xtr, ytr)

run_tests([
    ("held-out accuracy", round(
        accuracy_score(yte, model.predict(Xte)), 3), 1.0),
    ("out-of-bag estimate", round(model.oob_score_, 3), 0.945),
])

In [ ]:
def bootstrap_indices(n, draws):
    return [int(d * n) for d in draws]

def majority_vote(per_tree):
    n_trees = len(per_tree)
    return [1 if sum(col) > n_trees / 2 else 0 for col in zip(*per_tree)]

run_tests([
    ("draws to indices", bootstrap_indices(4, [0.1, 0.5, 0.99, 0.1]),
     [0, 2, 3, 0]),
    ("repeats are the point", bootstrap_indices(3, [0.0, 0.1, 0.2]),
     [0, 0, 0]),
    ("three trees, two birds", majority_vote([[1, 0], [1, 1], [0, 1]]),
     [1, 1]),
    ("outvoted quirk", majority_vote([[0], [0], [1]]), [0]),
    ("five trees agree in the end",
     majority_vote([[1, 0, 0], [0, 0, 1], [1, 1, 0], [1, 0, 0], [0, 0, 0]]),
     [1, 0, 0]),
])